# 01 -- data generation

plays thousands of games using heuristic expectimax and records
(encoded_board, cumulative_score_to_end) pairs for training the
neural network board evaluator.

board encoding: 16 floats, each cell = log2(value) / 17 (empty = 0).

output: `training_data.npz` containing `boards` and `scores` arrays.

In [ ]:
import numpy as np
import random
import math
import time

## game engine

In [ ]:
def clone(board):
    return [row[:] for row in board]

def get_empty_cells(board):
    return [(r, c) for r in range(4) for c in range(4) if board[r][c] == 0]

def spawn(board):
    b = clone(board)
    empty = get_empty_cells(b)
    if not empty:
        return b
    r, c = random.choice(empty)
    b[r][c] = 2 if random.random() < 0.9 else 4
    return b

def slide_row(row):
    score = 0
    vals = [v for v in row if v != 0]
    i = 0
    while i < len(vals) - 1:
        if vals[i] == vals[i + 1]:
            vals[i] *= 2
            score += vals[i]
            vals.pop(i + 1)
        i += 1
    while len(vals) < 4:
        vals.append(0)
    return vals, score

def move(board, direction):
    b = clone(board)
    total_score = 0
    moved = False
    for i in range(4):
        if direction == 0:
            row = [b[0][i], b[1][i], b[2][i], b[3][i]]
        elif direction == 1:
            row = [b[i][3], b[i][2], b[i][1], b[i][0]]
        elif direction == 2:
            row = [b[3][i], b[2][i], b[1][i], b[0][i]]
        else:
            row = [b[i][0], b[i][1], b[i][2], b[i][3]]
        new_row, score = slide_row(row)
        total_score += score
        for j in range(4):
            if direction == 0:   r2, c2 = j, i
            elif direction == 1: r2, c2 = i, 3 - j
            elif direction == 2: r2, c2 = 3 - j, i
            else:                r2, c2 = i, j
            if b[r2][c2] != new_row[j]:
                moved = True
            b[r2][c2] = new_row[j]
    return b, total_score, moved

def is_game_over(board):
    for r in range(4):
        for c in range(4):
            if board[r][c] == 0:
                return False
            if c < 3 and board[r][c] == board[r][c + 1]:
                return False
            if r < 3 and board[r][c] == board[r + 1][c]:
                return False
    return True

def create_board():
    board = [[0]*4 for _ in range(4)]
    board = spawn(board)
    board = spawn(board)
    return board

## heuristic evaluator + expectimax

In [ ]:
DEPTH = 3
PROB_THRESHOLD = 0.0001

def heuristic_evaluate(board):
    empty = 0
    max_val = 0
    mono = 0
    smooth = 0.0

    for r in range(4):
        for c in range(4):
            v = board[r][c]
            if v == 0:
                empty += 1
                continue
            log_v = math.log2(v)
            if v > max_val:
                max_val = v
            if c < 3 and board[r][c + 1] != 0:
                smooth -= abs(log_v - math.log2(board[r][c + 1]))
            if r < 3 and board[r + 1][c] != 0:
                smooth -= abs(log_v - math.log2(board[r + 1][c]))

    for r in range(4):
        inc = dec = 0
        for c in range(3):
            cur = 0 if board[r][c] == 0 else math.log2(board[r][c])
            nxt = 0 if board[r][c + 1] == 0 else math.log2(board[r][c + 1])
            if cur > nxt:
                dec += nxt - cur
            else:
                inc += cur - nxt
        mono += max(inc, dec)

    for c in range(4):
        inc = dec = 0
        for r in range(3):
            cur = 0 if board[r][c] == 0 else math.log2(board[r][c])
            nxt = 0 if board[r + 1][c] == 0 else math.log2(board[r + 1][c])
            if cur > nxt:
                dec += nxt - cur
            else:
                inc += cur - nxt
        mono += max(inc, dec)

    corners = [board[0][0], board[0][3], board[3][0], board[3][3]]
    corner = math.log2(max_val) if max_val in corners and max_val > 0 else 0

    return (empty * 2.7) + (mono * 1.0) + (smooth * 0.1) + (corner * 1.0)

def expectimax(board, depth, is_max, prob):
    if depth == 0 or is_game_over(board):
        return heuristic_evaluate(board)

    if is_max:
        best = float('-inf')
        for d in range(4):
            b, s, moved = move(board, d)
            if not moved:
                continue
            val = expectimax(b, depth - 1, False, prob)
            if val > best:
                best = val
        return best if best != float('-inf') else heuristic_evaluate(board)

    empty = get_empty_cells(board)
    if not empty:
        return heuristic_evaluate(board)

    avg = 0
    cell_prob = prob / len(empty)
    for r, c in empty:
        for val, tile_prob in [(2, 0.9), (4, 0.1)]:
            branch_prob = cell_prob * tile_prob
            if branch_prob < PROB_THRESHOLD:
                continue
            b = clone(board)
            b[r][c] = val
            avg += tile_prob * expectimax(b, depth - 1, True, branch_prob)
    return avg / len(empty)

def best_move(board):
    best_dir = -1
    best_val = float('-inf')
    for d in range(4):
        b, s, moved = move(board, d)
        if not moved:
            continue
        val = s + expectimax(b, DEPTH - 1, False, 1.0)
        if val > best_val:
            best_val = val
            best_dir = d
    return best_dir

## board encoding

In [ ]:
def encode_board(board):
    """encode a 4x4 board into 16 floats: log2(value)/17, empty=0."""
    encoded = []
    for r in range(4):
        for c in range(4):
            v = board[r][c]
            encoded.append(math.log2(v) / 17.0 if v > 0 else 0.0)
    return encoded

## play games and collect data

In [ ]:
NUM_GAMES = 500

all_boards = []
all_scores = []
game_stats = []

t_start = time.time()

for g in range(NUM_GAMES):
    board = create_board()
    game_boards = []
    game_move_scores = []
    total_score = 0

    while not is_game_over(board):
        game_boards.append(encode_board(board))
        game_move_scores.append(total_score)

        d = best_move(board)
        if d == -1:
            break
        board, score, moved = move(board, d)
        total_score += score
        board = spawn(board)

    # compute labels: cumulative score from each state to end
    for i, encoded in enumerate(game_boards):
        all_boards.append(encoded)
        all_scores.append(total_score - game_move_scores[i])

    max_tile = max(max(row) for row in board)
    game_stats.append((total_score, max_tile))

    if (g + 1) % 50 == 0:
        elapsed = time.time() - t_start
        avg_score = sum(s for s, _ in game_stats[-50:]) / 50
        print(f'game {g+1}/{NUM_GAMES} | avg score (last 50): {avg_score:.0f} | '
              f'elapsed: {elapsed:.0f}s | samples so far: {len(all_boards)}')

elapsed = time.time() - t_start
print(f'\ndone. {NUM_GAMES} games, {len(all_boards)} samples, {elapsed:.0f}s total')
print(f'avg score: {sum(s for s, _ in game_stats) / NUM_GAMES:.0f}')

tile_counts = {}
for _, mt in game_stats:
    tile_counts[mt] = tile_counts.get(mt, 0) + 1
print(f'max tile distribution: {dict(sorted(tile_counts.items()))}')

## save data

In [ ]:
boards_arr = np.array(all_boards, dtype=np.float32)
scores_arr = np.array(all_scores, dtype=np.float32)

print(f'boards shape: {boards_arr.shape}')
print(f'scores shape: {scores_arr.shape}')
print(f'score range: {scores_arr.min():.0f} - {scores_arr.max():.0f}')

np.savez_compressed('training_data.npz', boards=boards_arr, scores=scores_arr)
print('saved training_data.npz')

## package output

In [ ]:
import shutil, os
os.makedirs('output', exist_ok=True)
shutil.copy('training_data.npz', 'output/training_data.npz')
shutil.make_archive('01_data_generation_output', 'zip', 'output')
print('done. download 01_data_generation_output.zip')